# 081 — Curves of $\mu$ and $\sigma$ of $\ln D_{s5\text{-}95}$ given $AvgSA(0\text{–}3\,s)$, every site

For each of the 60 sites, how the distribution of significant duration $D_{s5\text{-}95}$
(OpenQuake IMT `RSD595`) depends on the conditioning intensity $x = AvgSA(0\text{–}3\,s)$,
drawn as two curves per site on **one common IML grid**:

$$
\mu(x) = E\left[\ln RSD595 \mid AvgSA = x\right], \qquad
\sigma(x) = SD\left[\ln RSD595 \mid AvgSA = x\right]
$$

These are site properties (hazard only), so both structures at a site (3- and 5-storey)
share the same pair of curves. They feed the fragility-correction models (nb 072 covariates
`d_ln_ds595_{mean,sigma}_unif`, nb 080 model `A8f_bc_hb`), which so far had these moments
only at the MSA-SS stripe IMLs; the curves let them be read at any IML
(`rsd595_curves.RSD595Curves`).

## Why a new disaggregation

nb 031's GCIM exists only at the record-selection stripes: 4–12 IMLs per site, a different
set at each site, nothing below 0.1645 g. The curves need every site on the same grid,
down to 0.01 g, so this notebook runs its own disaggregation child on nb 003's parent PSHA.

## Method

**Occurrence disaggregation.** At each site and grid level $x$ the child disaggregates at
$x$ and at $x + \delta x$, and the occurrence weights follow from the two exceedance
disaggregations (Bradley 2010, *EESD* 39:1321–1342, Eq. 7, p. 1326):

$$
w_k = P(m_k \mid X = x) \approx \frac{\nu_k(x) - \nu_k(x+\delta x)}{\nu(x) - \nu(x+\delta x)},
\qquad \nu(x) = \sum_k \nu_k(x), \qquad \delta x = 0.001\,x
$$

This is the same occurrence disaggregation as nbs 020/021 (`oq_multi_iml_disagg.collate_targets`),
and the GCIM below weights scenarios by it (`occurence = True`), exactly as nb 031 does for
record selection.

**Conditional moments within one scenario.** A scenario $k$ is one (TRT, M, R) bin.
Within it, $\ln RSD595$ and $\ln AvgSA$ are jointly normal, so given $\ln AvgSA = \ln x$:

$$
m_k = \mu_k + \rho_k\,\sigma_k\,\varepsilon_k, \qquad
s_k = \sigma_k\sqrt{1-\rho_k^2}
$$

where $\mu_k, \sigma_k$ come from the TRT's RSD595 model (Bahrampouri et al. 2021),
$\rho_k$ is the project's correlation between $\ln RSD595$ and $\ln AvgSA$ for that TRT, and
$\varepsilon_k = (\ln x - \mu^{AvgSA}_k)/\sigma^{AvgSA}_k$ is the epsilon of $x$ under the
AvgSA model **at the bin centre**. The disaggregation's own $\varepsilon$ bins are not used:
pickagm sums over them and recomputes $\varepsilon_k$.

**Mixing over scenarios** (law of total variance):

$$
\mu(x) = \sum_k w_k\, m_k, \qquad
\sigma^2(x) = \underbrace{\sum_k w_k\, s_k^2}_{\text{within scenarios}}
+ \underbrace{\sum_k w_k\, m_k^2 - \Big(\sum_k w_k\, m_k\Big)^2}_{\text{spread of scenario means}}
$$

This is pickagm's `gcim_distributions` (`_distribution_mean`, `_distribution_var`), run
with RSD595 as the only selection IM. Each IM is computed separately and the correlation is
looked up by label, so the result equals nb 031's RSD595 row exactly (checked: 544 of 544
stripes bit-identical).

Two things to keep in mind when reading the curves:

- The mixture is not normal. $\mu$ and $\sigma$ summarise it; they do not make it lognormal.
- As $x$ changes, both the weights $w_k$ (which scenarios dominate) and $\varepsilon_k$
  change, so the curves need not be monotonic, especially where the dominant TRT switches.

References (chapter and page numbers approximate, to be checked against the books):
Bradley (2010) *EESD* 39:1321–1342 (Eq. 7 p. 1326; GCIM conditional distributions §3);
Baker, Bradley & Stafford (2021) *Seismic Hazard and Risk Analysis*, Cambridge UP —
disaggregation (Ch. 7) and ground-motion selection (Ch. 10);
Benjamin & Cornell (1970, Dover reprint 2014) *Probability, Statistics, and Decision for
Civil Engineers*, conditional expectation and variance (Ch. 2, §2.4, pp. ~150–160).

## Dependencies

| | Notebook | What is used |
|---|---|---|
| upstream | `003` | parent PSHA `AvgSA_03_psha_eps4` (`max_sites_disagg = 60`, stored rupture contexts) |
| upstream | `004` | 4σ hazard curves (`AvgSA_03_hazard_curves_4sig`): MAFE for the stats and the zero-hazard test |
| upstream | `072` | regression dataset and stripes (lower-bound diagnostic in §1 only) |
| reused code | `020` / `021` / `031` | `oq_multi_iml_disagg`, `disagg_shards`, `gm_selection.calculate_gcim_distributions_for_sites` |
| downstream | `072`, `080` | the curves, through `rsd595_curves.RSD595Curves` |

**Isolation from record selection.** The child is named `AvgSA_03_disagg_eps4_rsdcurve`.
It keeps the `AvgSA_03_disagg_eps4_` prefix, so nb 021's provenance check covers it, but it
does not match nb 021's child pattern `…_imlgrid(_c\d+)?`, so nb 021 never collates it. All
outputs go to new paths (config keys `AvgSA_03_rsdcurve_*`, `rsd595_given_avgsa*`); nothing
that nbs 021/031–033 read is touched.

## Outputs

| config key | file |
|---|---|
| `proc_data.AvgSA_03_rsdcurve_imls` | the grid (one column, g) |
| `proc_data.AvgSA_03_rsdcurve_disagg_shards` | per-site shards, **slim**: ε bins summed, rows with $\nu_k = 0$ dropped (`rsd595_curves.slim_disagg_frame`) |
| `proc_data.AvgSA_03_rsdcurve_disagg_stats` | disagg stats, one row per (site, IML) |
| `proc_data.AvgSA_03_rsdcurve_excluded_imls` | zero-hazard IMLs per site |
| `proc_data.gcim_rsd595_curve_AvgSA_03` | RSD595-only GCIM at every kept (site, IML) |
| `proc_data.mu_lnRSD595_given_AvgSA_03` | $\mu$: index `site_id` 0–59, columns = grid IMLs in g, NaN above the site's ceiling |
| `proc_data.sigma_lnRSD595_given_AvgSA_03` | $\sigma$: same layout |
| `results.rsd595_given_avgsa_plots` | figures |

## Run order

1. **On the analysis machine:** §0–§2 (§2 launches OpenQuake). Run with `DRY_RUN = True`
   first to see what would run.
2. After the child has finished, still on the analysis machine (the datastore is local to
   it): §3, which writes the shards. From there on (§4–§6) any machine with the shards works.
3. An unchanged re-run recomputes nothing: the child is reused and every product is loaded
   from its cache.

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import json
import pickle
import re
from copy import deepcopy

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from openquake.hazardlib.imt import RSD595

from phd_project.config import config
from phd_project.scripts import cache_utils, disagg_shards, oq_runner, oqhelpers
from phd_project.scripts import oq_multi_iml_disagg as mid
from phd_project.scripts import rsd595_curves as rc

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
# The IM definition and truncation level of the main line.
IM = "03"
EPS = 4

# The common IML grid [g]: N_GRID log-spaced levels from IML_LO to IML_HI, plus ANCHORS.
# - IML_LO: nothing downstream needs IMLs below ~0.1 g (§1 diagnostic); 0.01 g leaves a
#   factor of 10 of margin at little cost.
# - IML_HI: 3.0 g is the last point of the parent's hazard-curve grid. Above it the
#   hazard-curve interpolation used for the stats and the zero-hazard test returns the
#   3.0 g value instead of zero, so the grid must not go higher (asserted in §1). Each
#   site's curve stops at its own ceiling anyway.
# - ANCHORS: IMLs that also exist in nb 017's record-selection grid, so the new
#   disaggregation can be compared with the existing one at the very same levels.
IML_LO, IML_HI, N_GRID = 0.01, 3.0, 37
ANCHORS = (0.27, 0.5, 0.95)

# Occurrence pair: every level x is also disaggregated at x * (1 + DELTA)
# (Bradley 2010, Eq. 7), as in nb 020.
DELTA = mid.DELTA_DEFAULT

# 0 = keep all realisations. Required: the mean disaggregation is only computed when
# more than one realisation is kept (as nb 020).
NUM_RLZS_DISAGG = 0

# Parent = nb 003's PSHA; child name keeps the AvgSA_03_disagg_eps4_ prefix (nb 021's
# provenance check covers it) but must NOT match nb 021's child pattern.
PARENT_NAME = mid.parent_name(IM, EPS)
CHILD_NAME = f"AvgSA_{IM}_disagg_eps{EPS}_rsdcurve"
CHILD_REGEX = rf"{re.escape(CHILD_NAME)}(_c\d+)?"
NB021_CHILD_REGEX = rf"AvgSA_{IM}_disagg_eps{EPS}_imlgrid(_c\d+)?"

# None = all 2 x 40 = 80 levels in one child (nb 020's child ran 76). An integer splits
# them over children of at most that many levels (CHILD_NAME_c00, _c01, ...) if
# OpenQuake cannot fit one child in memory. 40 gives two children of 20 IMLs.
MAX_LEVELS_PER_CHILD = None

# GCIM percentiles: computed by pickagm alongside mean and sigma; not used downstream.
PERCENTILES = [0.16, 0.5, 0.84]

# DRY_RUN:         write configs and report what would run, but launch nothing.
# FORCE_RERUN:     re-run the child even when its inputs are unchanged.
# NEW_WINDOW:      give the calculation its own console window (Windows; output is teed
#                  to WP1_DIR/logs/<name>.log either way).
# FORCE_RECOMPUTE: rebuild the shards, stats and GCIM even when their inputs are unchanged.
DRY_RUN = False
FORCE_RERUN = False
NEW_WINDOW = True
FORCE_RECOMPUTE = False
# -----------------------------------------------------------------------------

# Inputs
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]
PSHA_MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITE_MODEL_FP = cfg["hazard_models"]["eshm20_wp1_site_model"]
SITES_FP = cfg["results"]["selected_sites_csv"]
HC_FP = cfg["proc_data"]["AvgSA_03_hazard_curves_4sig"]
GMM_LT_FP = cfg["hazard_models"]["eshm20_AvgSA_03_median_lt"]

# Outputs
GRID_FP = cfg["proc_data"]["AvgSA_03_rsdcurve_imls"]
SHARD_DIR = cfg["proc_data"]["AvgSA_03_rsdcurve_disagg_shards"]
STATS_FP = cfg["proc_data"]["AvgSA_03_rsdcurve_disagg_stats"]
EXCL_FP = cfg["proc_data"]["AvgSA_03_rsdcurve_excluded_imls"]
GCIM_FP = cfg["proc_data"]["gcim_rsd595_curve_AvgSA_03"]
MU_FP = cfg["proc_data"]["mu_lnRSD595_given_AvgSA_03"]
SIGMA_FP = cfg["proc_data"]["sigma_lnRSD595_given_AvgSA_03"]
PLOT_DIR = cfg["results"]["rsd595_given_avgsa_plots"]

site_meta = pd.read_csv(SITES_FP)       # row i = datastore site id i (nb 020 check 1)
N_SITES = len(site_meta)
assert N_SITES == len(pd.read_csv(SITE_MODEL_FP)) == 60

print(f"{N_SITES} sites; parent {PARENT_NAME}; child {CHILD_NAME}")
print(f"DRY_RUN={DRY_RUN}, FORCE_RERUN={FORCE_RERUN}, NEW_WINDOW={NEW_WINDOW}, "
      f"FORCE_RECOMPUTE={FORCE_RECOMPUTE}")

## 1. The IML grid

**Lower end.** The diagnostic below gives, across the 120 rows of nb 072's dataset, the
lowest IMLs anything downstream uses: the lowest MSA-SS stripe and
$\theta\,e^{-3\beta}$ of the SS, FX and IDA fragilities (the IML at which each fragility is
three log-SDs below its median). The upper tail $\theta\,e^{+3\beta}$ is shown for
reference.

**Upper end.** Each site's curve stops at its own **ceiling**: the largest grid IML that is
not zero hazard, i.e. not above the site's ε-truncated maximum ground motion. The ceiling on
the hazard-curve grid (nb 017's `max_usable_iml`) is shown here; it is coarse (grid points a
factor of ~1.44 apart near the top, and for many sites just the end of the grid). The finer
ceiling from the disaggregation itself is added in §3.

In [ ]:
with open(HC_FP, "rb") as f:
    hcurves = pickle.load(f)
assert sorted(hcurves) == list(range(N_SITES))

# Lower-bound diagnostic over the 120 rows: the lowest IMLs anything downstream uses.
study_dir = config.bootstrap_roots(cfg)["study"]
dataset = pd.read_csv(study_dir / "fragility_regression_dataset_AvgSA_03.csv")
stripes = pd.read_csv(study_dir / "fragility_regression_stripes_AvgSA_03.csv")
assert len(dataset) == 120

diag = [{"quantity": "MSA-SS stripe IML", "min [g]": stripes["iml"].min(),
         "5th pct [g]": np.nan, "max [g]": stripes["iml"].max()}]
for arm, label in [("msa_ss", "MSA-SS"), ("msa_fx", "MSA-FX"), ("ida_fx", "IDA-FX")]:
    theta, beta = dataset[f"{arm}_theta"], dataset[f"{arm}_beta"]
    lo, hi = theta * np.exp(-3 * beta), theta * np.exp(3 * beta)
    diag.append({"quantity": f"theta exp(-3 beta), {label}", "min [g]": lo.min(),
                 "5th pct [g]": lo.quantile(0.05), "max [g]": lo.max()})
    diag.append({"quantity": f"theta exp(+3 beta), {label}", "min [g]": hi.min(),
                 "5th pct [g]": np.nan, "max [g]": hi.max()})
display(pd.DataFrame(diag).set_index("quantity").round(4))

In [ ]:
grid = rc.build_iml_grid(IML_LO, IML_HI, N_GRID, ANCHORS)

# The grid must stay within every site's hazard-curve grid (see IML_HI above).
hc_top = min(float(np.asarray(hcurves[s]["AvgSA"]["mean"])[-1, 0]) for s in hcurves)
assert grid[-1] <= hc_top + 1e-12, f"grid top {grid[-1]} g above the hazard-curve grid ({hc_top} g)"

# Written in nb 017's IML-file format; only rewritten when it changes (it is fingerprinted).
rc.write_iml_grid(grid, GRID_FP)
assert grid == [float(oq_runner._iml_str(v)) for v in oq_runner.load_imls(GRID_FP)]

ceil_hc = rc.hazard_curve_ceilings(hcurves)
print(f"{len(grid)} grid IMLs ({N_GRID} log-spaced, ratio {grid[1] / grid[0]:.4f}, "
      f"+ anchors {list(ANCHORS)}); {sum(g < 0.1645 for g in grid)} below 0.1645 g:")
print(grid)
print("\nceiling on the hazard-curve grid -> number of sites")
print(ceil_hc.value_counts().sort_index().to_string())

## 2. The disaggregation child

Every site gets every grid IML at $x$ and $x+\delta x$: 40 × 2 = 80 levels per site
($N = 60$, $M = 1$, $P = 80$). Same call pattern as nb 020 §3: the parent is checked
(`require_parent`), the levels table and ini are written next to nb 020's in
`hazard_models/eshm20/wp1/`, and `run_or_reuse_child` runs the child, or reuses it if its
inputs (ini, levels, parent, engine) are unchanged, and records it in `disagg_manifest.json`.

Changing the grid or `DELTA` re-runs the child. That is fine here: nothing caches per-stripe
results on this disaggregation, unlike nb 020's.

In [ ]:
# nb 021 collates every child matching its pattern; this child must not be one of them.
assert re.fullmatch(NB021_CHILD_REGEX, CHILD_NAME) is None

parent_calc_id = mid.require_parent(PSHA_MANIFEST_FP, PARENT_NAME, N_SITES)
print(f"parent {PARENT_NAME}: calc_id = {parent_calc_id}")

targets = pd.DataFrame([(s, k, k) for s in range(N_SITES) for k in grid],
                       columns=["site", "key", "x"])
chunks = mid.split_targets(targets, MAX_LEVELS_PER_CHILD)

child_calc_ids = {}
if parent_calc_id is None:
    print("parent not available - the child ini needs its calc_id; nothing written")
else:
    for i, chunk in enumerate(chunks):
        name = CHILD_NAME if len(chunks) == 1 else f"{CHILD_NAME}_c{i:02d}"
        table = mid.build_levels_table(chunk, DELTA)
        ini = mid.write_child_config(
            WP1_DIR / f"config_{name}.ini", parent_calc_id=parent_calc_id,
            parent_name=PARENT_NAME, n_levels=int(table.p.max()) + 1,
            description=mid.child_description(name),
            levels_file=f"{mid.levels_file_stem(name)}.npy",
            im=IM, truncation_level=EPS, num_rlzs_disagg=NUM_RLZS_DISAGG)
        child_calc_ids[name] = mid.run_or_reuse_child(
            name, ini, table, N_SITES, WP1_DIR, MANIFEST_FP, parent_name=PARENT_NAME,
            parent_manifest_fp=PSHA_MANIFEST_FP,
            force_rerun=FORCE_RERUN, dry_run=DRY_RUN, new_window=NEW_WINDOW)
        print(f"  {name}: {chunk['key'].nunique()} IMLs, {table.p.max() + 1} levels per site "
              f"-> calc_id {child_calc_ids[name]}")
child_calc_ids

---
## ⛔ RUN BARRIER

The cells above launch OpenQuake on the analysis machine. The cells below need the finished
child datastore (§3) or the shards written from it (§4–§6).

---

## 3. Collate to shards, stats and exclusions

As nb 021 §2, for this child only. `mid.collate_targets` forms the occurrence column from
each $x$ / $x+\delta x$ pair, then `rc.slim_disagg_frame` sums each frame over its ε bins
and drops bins with $\nu_k = 0$ — one site at a time, so only one site's full-size frames
are ever in memory. This changes no result: pickagm sums over the ε bins and drops
zero-weight rows anyway, and the stats use only TRT, M, R, $\nu_k$ and $P(m \mid X > x)$.

The child(ren) in the manifest must carry the grid exactly, once each; if the grid or
`MAX_LEVELS_PER_CHILD` changed since an earlier run, remove the superseded
`…_rsdcurve*` entries from `disagg_manifest.json` and re-run §2.

**Zero hazard.** An IML is excluded when the hazard curve gives zero rate there or no
disaggregation bin can exceed it (`oqhelpers.get_excluded_imls`, as nb 021). Each site's
**ceiling** is its largest grid IML not excluded; its curves stop there.

In [ ]:
from openquake.commonlib.datastore import read

disagg_manifest = oq_runner.load_manifest(MANIFEST_FP)
psha_manifest = oq_runner.load_manifest(PSHA_MANIFEST_FP)
children = mid.child_entries(disagg_manifest, CHILD_REGEX)
if not children:
    raise KeyError(f"no {CHILD_NAME} child in {MANIFEST_FP.name} - run §2 first")

# The children must carry exactly the grid, each IML once.
carried = {}
for name, e in children.items():
    for k in mid.child_keys(WP1_DIR / e["levels_csv"]):
        if k in carried:
            raise ValueError(f"IML {k} is carried by both {carried[k]} and {name}")
        carried[k] = name
if sorted(carried) != grid:
    raise ValueError(f"the {CHILD_NAME} children do not carry the grid exactly: missing "
                     f"{sorted(set(grid) - set(carried))}, extra "
                     f"{sorted(set(carried) - set(grid))}. Remove superseded entries from "
                     f"the manifest and re-run §2.")

# The hazard curves must come from the same source model, GMM logic tree and site model.
oqhelpers.assert_shared_provenance(disagg_manifest, psha_manifest, IM, EPS)

# Provenance of the shards (and the stats): the children, the grid, the site metadata and
# hazard curves, and the frame transform (switching it would change every shard).
fp_inputs = {}
for name, e in children.items():
    fp_inputs[f"config_{name}"] = WP1_DIR / e["config"]
    fp_inputs[f"levels_{name}"] = WP1_DIR / e["levels_csv"]
    fp_inputs[f"calc_id_{name}"] = str(e["calc_id"])
    fp_inputs[f"parent_calc_id_{name}"] = str(e["parent_calc_id"])
fp_inputs["sites_csv"] = SITES_FP
fp_inputs["iml_grid"] = GRID_FP
fp_inputs["hazard_curves"] = HC_FP
fp_inputs["frame_transform"] = "rsd595_curves.slim_disagg_frame: eps summed, nu_m == 0 dropped"
fp_dict = cache_utils.fingerprint(**fp_inputs)


def collate():
    # One child at a time: open its datastore, check its levels file against its own
    # hmap3, form the occurrence from each x / x+dx pair, slim each frame, close.
    parts = []
    for name, e in children.items():
        dstore = read(int(e["calc_id"]))
        try:
            table = mid.read_levels_table(WP1_DIR / e["levels_csv"])
            parts.append(mid.collate_targets(dstore, table, mid.DISAGG_TYPE,
                                             frame_fn=rc.slim_disagg_frame))
        finally:
            dstore.close()
    data = mid.merge_collations(parts)
    assert sorted(data) == list(range(N_SITES)), "a site is missing"
    for site, by_imt in data.items():
        if sorted(by_imt["AvgSA"]) != grid:
            raise ValueError(f"site {site}: the IMLs do not match the grid")
    return data


disagg_data = disagg_shards.load_or_compute_shards(
    SHARD_DIR, fp_dict, collate, force_recompute=FORCE_RECOMPUTE)

# Stats: as nb 021, with the zero-hazard rule named in the fingerprint (it is code).
stats_fp_dict = {**fp_dict, **cache_utils.fingerprint(
    zero_hazard_rule="mafe <= 0 or no non-zero disagg bin")}
disagg_stats = cache_utils.load_or_compute(
    STATS_FP, stats_fp_dict,
    lambda: oqhelpers.get_iml_disagg_stats(disagg_data, site_meta, hcurves),
    force_recompute=FORCE_RECOMPUTE)

excluded = oqhelpers.get_excluded_imls(disagg_data, hcurves)
with open(EXCL_FP, "w") as f:
    json.dump({str(s): imls for s, imls in excluded.items()}, f, indent=2)

# The occurrence weights must sum to 1 at every kept (site, IML).
occ_err = rc.occurrence_sum_error(disagg_data, excluded)
assert occ_err < 1e-12, f"occurrence weights do not sum to 1 (max error {occ_err})"

# Ceilings: from the disaggregation (used) next to the hazard-curve one (coarse).
# disagg_ceilings refuses a zero-hazard IML below a kept one (a hole in a curve).
ceilings = rc.disagg_ceilings(grid, excluded, range(N_SITES)).join(ceil_hc)
ceilings["seismicity"] = site_meta["seismicity"]
print(f"{len(disagg_stats)} stat rows; {sum(len(v) for v in excluded.values())} "
      f"zero-hazard (site, IML) excluded; max |sum P(m|X=x) - 1| = {occ_err:.1e}")
display(ceilings)

## 4. GCIM of RSD595 at every (site, grid IML)

The selection context is nb 031's (`setup_AvgSA03_gcim_gm_selection`, loading no shards),
with RSD595 as the only selection IM. Everything else is unchanged: the same GMMs
(Bahrampouri et al. 2021 for RSD595, per TRT), the project correlation models, the same
rupture-context builder and `occurence = True`. Keys are every (site, IML) of the new shards
that is not zero hazard. About 2 400 keys at ~0.03 s each.

In [ ]:
from phd_project.scripts.WP1_ground_motion_set import setup_AvgSA03_gm_selection as gm_setup
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    _selection_ctx_fingerprint_inputs,
    calculate_gcim_distributions_for_sites,
)

_, _, site_model, basic_ctx, _ = gm_setup.setup_AvgSA03_gcim_gm_selection(sites=())
ctx = deepcopy(basic_ctx)
ctx["selection_imts"] = [RSD595()]
ctx["imt_weights"] = np.array([1.0])
assert ctx["occurence"] is True, "the GCIM must weight scenarios by P(m|X=x)"

gcim_keys = sorted((s, k) for s, by_imt in disagg_data.items() for k in by_imt["AvgSA"]
                   if k not in set(excluded.get(s, [])))

gcim_fp_dict = cache_utils.fingerprint(
    disagg_shards=disagg_shards.shards_digest(SHARD_DIR),
    excluded_imls=json.dumps({str(s): v for s, v in sorted(excluded.items())}),
    site_model_file=SITE_MODEL_FP,
    gmm_lt_file=GMM_LT_FP,
    percentiles=tuple(PERCENTILES),
    **_selection_ctx_fingerprint_inputs(ctx))

gcim = cache_utils.load_or_compute(
    GCIM_FP, gcim_fp_dict,
    lambda: calculate_gcim_distributions_for_sites(
        {k: disagg_data[k[0]]["AvgSA"][k[1]] for k in gcim_keys},
        disagg_stats, site_model, ctx, PERCENTILES),
    force_recompute=FORCE_RECOMPUTE)
assert sorted(gcim) == gcim_keys
print(f"GCIM at {len(gcim)} (site, IML)")

## 5. The curve tables

Two CSV files in the same wide layout: index `site_id` (0–59), one column per grid IML in g
(ascending, exact 6-significant-figure keys), values $\mu$ or $\sigma$ in log units. A cell
is empty (NaN) above the site's ceiling; nothing is filled or extrapolated. Read them with
`rsd595_curves.read_curves` (plain `pd.read_csv` returns the column labels as text), or
read the curves at any IML with `rsd595_curves.RSD595Curves`, which interpolates linearly
in $\ln x$ between grid points and returns NaN outside a site's range.

In [ ]:
mu, sigma = rc.curves_from_gcim(gcim, grid, range(N_SITES))

# Every site has values exactly at its first n_kept grid IMLs, i.e. up to its ceiling,
# with no hole below it.
for s in range(N_SITES):
    have = mu.loc[s].notna().to_numpy()
    n = int(ceilings.loc[s, "n_kept"])
    assert have[:n].all() and not have[n:].any(), f"site {s}: values do not match its ceiling"
assert (mu.isna() == sigma.isna()).all().all()

rc.write_curves(mu, MU_FP)
rc.write_curves(sigma, SIGMA_FP)

# Round trip: what downstream reads back is exactly what was computed.
for df, fp in [(mu, MU_FP), (sigma, SIGMA_FP)]:
    pd.testing.assert_frame_equal(rc.read_curves(fp), df, check_exact=True, check_names=False)
curves = rc.RSD595Curves(MU_FP, SIGMA_FP)
print(f"wrote {MU_FP.name} and {SIGMA_FP.name}: {mu.shape[0]} sites x {mu.shape[1]} IMLs")
display(mu.iloc[:5, ::6].round(3))

## 6. Plots

Drawn from the two CSV files (through `curves`), so the figures show exactly what was
written. Plain curves on a log IML axis over the full grid range, with the same y-limits in
every panel of a figure so sites can be compared directly. Colours: categorical slots 1 and
2 of the project's reference palette for the two seismicity groups.

In [ ]:
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MU_T, SIG_T = curves.mu_table, curves.sigma_table
X = np.asarray(MU_T.columns, float)
XLIM = (X[0], X[-1])
SEIS = site_meta["seismicity"]
SEIS_COLOURS = {"lowmod": "#2a78d6", "high": "#eb6834"}
SEIS_LABELS = {"lowmod": "low–moderate seismicity", "high": "high seismicity"}
X_LABEL = r"$AvgSA(0\text{–}3\,s)$ [g]"
MU_LABEL = r"$E[\ln D_{s5\text{-}95} \mid AvgSA]$ [ln s]"
SIG_LABEL = r"$SD[\ln D_{s5\text{-}95} \mid AvgSA]$"


def common_ylim(table, pad=0.04):
    # One y-range for all sites, so panels are comparable.
    lo, hi = np.nanmin(table.to_numpy()), np.nanmax(table.to_numpy())
    return lo - pad * (hi - lo), hi + pad * (hi - lo)


def style_axis(ax):
    # Log IML axis over the full grid; recessive grid and frame.
    ax.set_xscale("log")
    ax.set_xlim(XLIM)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.grid(True, which="major", color="0.88", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def save_fig(fig, name):
    fig.savefig(PLOT_DIR / name, dpi=200, bbox_inches="tight", facecolor="w")

In [ ]:
# Per site: 6 x 10 small multiples, one figure for mu and one for sigma.
def small_multiples(table, ylabel, fname):
    fig, axes = plt.subplots(6, 10, figsize=(20, 12.5), sharex=True, sharey=True)
    ylim = common_ylim(table)
    for s, ax in zip(table.index, axes.flat):
        ax.plot(X, table.loc[s].to_numpy(float), color="0.2", lw=1.4)
        style_axis(ax)
        ax.set_ylim(ylim)
        ax.set_title(f"site {s}", fontsize=8, pad=2)
        ax.tick_params(labelsize=7)
    fig.supxlabel(X_LABEL)
    fig.supylabel(ylabel, x=0.002)
    # Leave a strip on the left and bottom for the shared labels, clear of the tick labels.
    fig.tight_layout(rect=(0.012, 0.01, 1, 1))
    save_fig(fig, fname)
    plt.show()


small_multiples(MU_T, MU_LABEL, "mu_lnRSD595_given_AvgSA_03_per_site.png")
small_multiples(SIG_T, SIG_LABEL, "sigma_lnRSD595_given_AvgSA_03_per_site.png")

In [ ]:
# All sites together, coloured by seismicity group.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, table, ylabel in [(axes[0], MU_T, MU_LABEL), (axes[1], SIG_T, SIG_LABEL)]:
    for s in table.index:
        ax.plot(X, table.loc[s].to_numpy(float), color=SEIS_COLOURS[SEIS[s]], lw=1.0, alpha=0.8)
    style_axis(ax)
    ax.set_ylim(common_ylim(table))
    ax.set_xlabel(X_LABEL)
    ax.set_ylabel(ylabel)
handles = [Line2D([], [], color=SEIS_COLOURS[k], lw=2, label=f"{SEIS_LABELS[k]} ({(SEIS == k).sum()} sites)")
           for k in SEIS_COLOURS]
axes[0].legend(handles=handles, frameon=False, fontsize=9)
fig.tight_layout()
save_fig(fig, "rsd595_given_AvgSA_03_all_sites.png")
plt.show()

In [ ]:
# Variation between sites: the across-site SD of mu and of sigma at each IML, and below them
# the number of sites the SD is taken over (sites drop out above their ceilings). The count
# has its own panel rather than a second y-axis on the same plot.
n_sites = MU_T.notna().sum(axis=0).to_numpy()
sd_mu = MU_T.std(axis=0, ddof=1).where(n_sites >= 2).to_numpy(float)
sd_sig = SIG_T.std(axis=0, ddof=1).where(n_sites >= 2).to_numpy(float)

fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 3, 1.4]})
axes[0].plot(X, sd_mu, color="0.2", lw=1.6)
axes[0].set_ylabel(r"SD across sites of $\mu$ [ln s]")
axes[1].plot(X, sd_sig, color="0.2", lw=1.6)
axes[1].set_ylabel(r"SD across sites of $\sigma$")
axes[2].step(X, n_sites, where="mid", color="0.45", lw=1.4)
axes[2].set_ylim(0, N_SITES * 1.08)
axes[2].set_ylabel("sites")
axes[2].set_xlabel(X_LABEL)
for ax in axes:
    style_axis(ax)
for ax in axes[:2]:
    ax.set_ylim(bottom=0)
fig.tight_layout()
save_fig(fig, "rsd595_given_AvgSA_03_between_site_variation.png")
plt.show()

## 7. Summary

*To be written once the curves have been computed and the plots looked at.*